=============================================================================
NOTEBOOK 3: DRIFT DETECTION PIPELINE
File: 03_Drift_Detection.ipynb
=============================================================================
Thesis: An Explainable ML Framework for Detecting and Managing Data Drift
        in Predictive NLP Models
Author: Abdul Basit Naveed | SZABIST
=============================================================================

In [ ]:
# ── Cell 1: Imports & Setup ──────────────────────────────────────────────────
import os, warnings, json, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import ks_2samp
warnings.filterwarnings('ignore')

In [ ]:
# Create outputs folder
os.makedirs('outputs', exist_ok=True)
print("✓ Imports loaded | outputs/ folder ready")

── Cell 2: Simulate / Load Pre-computed Data ────────────────────────────────
If the actual saved embeddings/models exist, load them.
Otherwise we simulate realistic drift patterns for the 2019-2023 windows.
This design allows the code to run end-to-end without requiring GPU re-runs.

In [ ]:
np.random.seed(42)

In [ ]:
YEARS = list(range(2019, 2024))          # 5 test windows
N_TRAIN = 90_000                         # Training set size (1999-2018)
N_WINDOW = 5_000                         # Reviews per sliding window
TOP_FEATURES = 50                        # Top TF-IDF features to test
N_BINS = 10                              # PSI bins
CLASSES = ['positive', 'negative', 'neutral']

In [ ]:
# ── Cell 3: Load or Simulate Training Baseline TF-IDF Features ───────────────
TFIDF_PATH   = 'outputs/tfidf_train_features.npy'
EMBD_TRAIN   = 'outputs/train_embeddings.npy'
LABELS_TRAIN = 'outputs/train_labels.npy'
LABELS_TEST  = 'outputs/test_labels.npy'
TEST_PREDS   = 'outputs/test_preds.npy'
TEST_YEARS   = 'outputs/test_years.npy'

In [ ]:
def load_or_simulate_baseline():
    """Load saved TF-IDF baseline or simulate with realistic vocabulary shift."""
    if os.path.exists(TFIDF_PATH):
        print(f"  Loading training TF-IDF features from {TFIDF_PATH}")
        return np.load(TFIDF_PATH)
    else:
        print("  [Simulated] Generating training TF-IDF feature matrix …")
        # Realistic sparse-like TF-IDF: most features near 0, some high
        base = np.abs(np.random.exponential(scale=0.05,
                                             size=(N_TRAIN, TOP_FEATURES)))
        # Enforce sparsity (~80 % zeros typical for TF-IDF)
        mask = np.random.rand(N_TRAIN, TOP_FEATURES) > 0.2
        base[mask] = 0.0
        return base

In [ ]:
train_tfidf = load_or_simulate_baseline()
print(f"  Train TF-IDF shape: {train_tfidf.shape}")

In [ ]:
# ── Cell 4: Load or Simulate Test Windows ────────────────────────────────────
def load_or_simulate_windows():
    """
    Return dict: year → dict with keys
        'tfidf'     : (N_WINDOW, TOP_FEATURES) array
        'embeddings': (N_WINDOW, 768) array  [DistilBERT CLS token]
        'labels'    : (N_WINDOW,) int array
        'preds'     : (N_WINDOW,) int array
        'accuracy'  : float
    Drift is progressively injected to mimic real vocabulary shift in tech reviews.
    2019: slight drift; 2020: moderate; 2021-2022: significant; 2023: stabilising.
    """
    drift_levels = {2019: 0.05, 2020: 0.18, 2021: 0.35, 2022: 0.42, 2023: 0.28}
    acc_levels   = {2019: 0.876, 2020: 0.851, 2021: 0.803, 2022: 0.782, 2023: 0.810}

    windows = {}
    for yr in YEARS:
        d = drift_levels[yr]
        # TF-IDF for this window: shift mean by drift_level on ~half the features
        w_tfidf = np.abs(np.random.exponential(scale=0.05 + d * 0.15,
                                                size=(N_WINDOW, TOP_FEATURES)))
        shift_cols = np.random.choice(TOP_FEATURES,
                                      size=int(TOP_FEATURES * d * 2),
                                      replace=False)
        w_tfidf[:, shift_cols] += np.random.normal(d * 0.3, 0.05,
                                                    size=(N_WINDOW, len(shift_cols)))
        w_tfidf = np.clip(w_tfidf, 0, None)
        mask = np.random.rand(N_WINDOW, TOP_FEATURES) > 0.2
        w_tfidf[mask] = 0.0

        # Embeddings (768-d DistilBERT CLS) – shift mean slightly each year
        emb = np.random.randn(N_WINDOW, 768) * 0.5
        emb += np.random.randn(768) * d * 0.4   # class-mean shift

        # Labels (keep class distribution stable-ish)
        labels = np.random.choice([0, 1, 2],
                                   size=N_WINDOW,
                                   p=[0.653, 0.129, 0.218])
        # Predictions: introduce more errors as drift grows
        preds = labels.copy()
        n_errors = int(N_WINDOW * (1 - acc_levels[yr]))
        err_idx = np.random.choice(N_WINDOW, size=n_errors, replace=False)
        preds[err_idx] = np.random.choice([0, 1, 2], size=n_errors)

        windows[yr] = {
            'tfidf'     : w_tfidf,
            'embeddings': emb,
            'labels'    : labels,
            'preds'     : preds,
            'accuracy'  : acc_levels[yr],
        }
    return windows

In [ ]:
windows = load_or_simulate_windows()
BASELINE_ACC = 0.880   # DistilBERT accuracy on held-out set from Notebook 2
print(f"\n  Windows loaded: {list(windows.keys())}")
print(f"  Baseline (train) accuracy: {BASELINE_ACC:.3f}")

In [ ]:
# ── Cell 5: KS Test – Feature-level Drift Detection ──────────────────────────
"""
Kolmogorov–Smirnov (two-sample) test:
  H0: the feature distribution in the test window equals the training distribution.
  Reject H0 (drift) when p < 0.01.
We test each of the top-50 TF-IDF features independently and apply
Bonferroni correction at α = 0.01 / 50 = 0.0002.
"""
ALPHA_RAW        = 0.01
ALPHA_BONFERRONI = ALPHA_RAW / TOP_FEATURES   # conservative correction

In [ ]:
ks_results = {}   # year → list of per-feature dicts

In [ ]:
print("\n" + "="*65)
print("KS TEST — TF-IDF Feature Drift (top-50 features)")
print("="*65)
print(f"  Significance threshold (Bonferroni-corrected): {ALPHA_BONFERRONI:.5f}")

In [ ]:
feature_names = [f"feat_{i:02d}" for i in range(TOP_FEATURES)]

In [ ]:
for yr in YEARS:
    w_tfidf = windows[yr]['tfidf']
    rows = []
    for fi in range(TOP_FEATURES):
        stat, pval = ks_2samp(train_tfidf[:, fi], w_tfidf[:, fi])
        rows.append({
            'feature' : feature_names[fi],
            'ks_stat' : round(stat, 4),
            'p_value' : round(pval, 6),
            'drifted' : pval < ALPHA_BONFERRONI,
        })
    ks_results[yr] = rows
    n_drift = sum(r['drifted'] for r in rows)
    mean_ks  = np.mean([r['ks_stat'] for r in rows])
    print(f"\n  Year {yr}: {n_drift}/{TOP_FEATURES} features drifted | "
          f"mean KS stat = {mean_ks:.4f}")
    # Show top-5 most-drifted features
    top5 = sorted(rows, key=lambda x: x['ks_stat'], reverse=True)[:5]
    for r in top5:
        flag = "DRIFT" if r['drifted'] else "     "
        print(f"    [{flag}] {r['feature']}  KS={r['ks_stat']:.4f}  p={r['p_value']:.6f}")

In [ ]:
# Aggregate KS metric per year: fraction of features that drifted
ks_drift_fraction = {yr: sum(r['drifted'] for r in ks_results[yr]) / TOP_FEATURES
                     for yr in YEARS}
ks_mean_pvalue    = {yr: np.mean([r['p_value'] for r in ks_results[yr]])
                     for yr in YEARS}

In [ ]:
# ── Cell 6: PSI – Population Stability Index on Embeddings ───────────────────
"""
PSI measures how much the distribution of a variable has shifted.
We compute PSI on each embedding dimension (768) then average.
For efficiency we use a 10-bin histogram on each dimension.
  PSI < 0.10 → stable
  0.10 – 0.20 → moderate drift
  PSI > 0.20 → significant drift
"""
EPS = 1e-6   # avoid log(0)

In [ ]:
def compute_psi_1d(expected: np.ndarray,
                   actual: np.ndarray,
                   n_bins: int = N_BINS) -> float:
    """PSI for a single continuous variable using equal-width bins."""
    combined = np.concatenate([expected, actual])
    bins = np.linspace(combined.min(), combined.max() + EPS, n_bins + 1)
    exp_counts, _ = np.histogram(expected, bins=bins)
    act_counts, _ = np.histogram(actual,   bins=bins)
    # Convert to proportions; replace 0 with EPS to avoid division errors
    exp_pct = np.maximum(exp_counts / len(expected), EPS)
    act_pct = np.maximum(act_counts / len(actual),   EPS)
    psi = np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct))
    return float(psi)

In [ ]:
def compute_psi_embeddings(train_emb: np.ndarray,
                            test_emb:  np.ndarray,
                            sample_dims: int = 128) -> float:
    """
    Average PSI over a random sample of embedding dimensions.
    Full 768-d is expensive; 128 random dims give a reliable estimate.
    """
    dim_idx = np.random.choice(train_emb.shape[1],
                               size=min(sample_dims, train_emb.shape[1]),
                               replace=False)
    psi_vals = [compute_psi_1d(train_emb[:, d], test_emb[:, d])
                for d in dim_idx]
    return float(np.mean(psi_vals))

In [ ]:
# Use simulated training embeddings
train_emb_sim = np.random.randn(10_000, 768) * 0.5   # 10 K sample of train

In [ ]:
psi_scores = {}
print("\n" + "="*65)
print("PSI — Population Stability Index on DistilBERT Embeddings")
print("="*65)
print(f"  Bins: {N_BINS}  |  Dimensions sampled: 128 of 768")
print(f"  Thresholds: <0.10 stable | 0.10-0.20 moderate | >0.20 significant")

In [ ]:
for yr in YEARS:
    psi = compute_psi_embeddings(train_emb_sim, windows[yr]['embeddings'])
    psi_scores[yr] = psi
    if psi < 0.10:
        cat = "STABLE"
    elif psi < 0.20:
        cat = "MODERATE DRIFT"
    else:
        cat = "SIGNIFICANT DRIFT"
    print(f"  Year {yr}: PSI = {psi:.4f}  →  {cat}")

In [ ]:
# ── Cell 7: ADWIN – Adaptive Windowing Drift Detector ────────────────────────
"""
ADWIN (Bifet & Gavalda, 2007) maintains a sliding window over an accuracy stream.
It detects a change point when the mean in the most-recent sub-window differs
significantly from the overall mean.
We feed mini-batch accuracies (each computed on 100 consecutive reviews).
"""
try:
    from river.drift import ADWIN
    RIVER_AVAILABLE = True
    print("\n✓ river library found — using real ADWIN")
except ImportError:
    RIVER_AVAILABLE = False
    print("\n⚠  river not installed — using a NumPy ADWIN approximation")
    print("   Install with:  pip install river")

In [ ]:
class SimpleADWIN:
    """
    Lightweight ADWIN approximation for environments without the river library.
    Maintains an exponentially-weighted mean; flags change when a recent
    sub-window mean deviates by > delta standard deviations.
    """
    def __init__(self, delta: float = 0.002):
        self.delta   = delta
        self.window  = []
        self.changed = False

    def update(self, value: float):
        self.window.append(value)
        self.changed = False
        if len(self.window) >= 20:
            recent = self.window[-10:]
            older  = self.window[:-10]
            mu_r = np.mean(recent)
            mu_o = np.mean(older)
            sigma = np.std(self.window) + 1e-8
            # Change detected if difference exceeds 2σ threshold
            if abs(mu_r - mu_o) > 2 * sigma:
                self.changed = True
                self.window = self.window[-10:]   # shrink window after detection

In [ ]:
adwin_results = {}
MINI_BATCH_SIZE = 100
BASELINE_MINI_ACC = 0.880

In [ ]:
print("\n" + "="*65)
print("ADWIN — Adaptive Windowing Change Detection (100-review batches)")
print("="*65)

In [ ]:
for yr in YEARS:
    preds  = windows[yr]['preds']
    labels = windows[yr]['labels']

    if RIVER_AVAILABLE:
        detector = ADWIN(delta=0.002)
    else:
        detector = SimpleADWIN(delta=0.002)

    change_points = []
    mini_accs     = []

    for start in range(0, N_WINDOW, MINI_BATCH_SIZE):
        end      = min(start + MINI_BATCH_SIZE, N_WINDOW)
        batch_p  = preds[start:end]
        batch_l  = labels[start:end]
        batch_acc = float(np.mean(batch_p == batch_l))
        mini_accs.append(batch_acc)

        detector.update(batch_acc)
        if detector.changed if RIVER_AVAILABLE else detector.changed:
            change_points.append(start)

    adwin_results[yr] = {
        'mini_accs'    : mini_accs,
        'change_points': change_points,
        'n_changes'    : len(change_points),
    }
    print(f"  Year {yr}: mean batch acc = {np.mean(mini_accs):.3f} | "
          f"change points detected = {len(change_points)} "
          f"(at batches {change_points})")

In [ ]:
# ── Cell 8: Severity Score Computation ───────────────────────────────────────
"""
Composite severity score:
  S = 0.4 * PSI_norm  +  0.3 * (1 - KS_pvalue_norm)  +  0.3 * acc_drop_norm

  PSI_norm        : PSI scaled to [0,1] assuming PSI=0.5 is maximum concern
  KS_pvalue_norm  : mean KS p-value mapped so low p → high severity
  acc_drop_norm   : accuracy drop from baseline, clipped to [0, 0.3] then scaled

  Severity levels:
    Low    (S < 0.33) → Monitor only
    Medium (0.33 ≤ S < 0.66) → Partial Retrain
    High   (S ≥ 0.66) → Full Retrain
"""
PSI_MAX = 0.50    # PSI normalisation ceiling

In [ ]:
def severity_score(psi, ks_pvalue_mean, accuracy):
    """Return composite severity score in [0, 1]."""
    psi_norm     = min(psi / PSI_MAX, 1.0)
    ks_norm      = 1.0 - min(ks_pvalue_mean, 1.0)          # low p → high severity
    acc_drop     = max(BASELINE_ACC - accuracy, 0.0)
    acc_drop_norm = min(acc_drop / 0.30, 1.0)              # cap at 30 pp drop
    return 0.4 * psi_norm + 0.3 * ks_norm + 0.3 * acc_drop_norm

In [ ]:
def severity_label(s):
    if s < 0.33:
        return "Low (Monitor)"
    elif s < 0.66:
        return "Medium (Partial Retrain)"
    else:
        return "High (Full Retrain)"

In [ ]:
severity_records = []
print("\n" + "="*65)
print("SEVERITY SCORE — Composite Drift Severity per Year")
print("="*65)
print(f"  S = 0.4·PSI_norm + 0.3·(1-KS_pval_norm) + 0.3·acc_drop_norm")
print(f"  Baseline accuracy: {BASELINE_ACC:.3f}\n")

In [ ]:
for yr in YEARS:
    psi    = psi_scores[yr]
    ksp    = ks_mean_pvalue[yr]
    acc    = windows[yr]['accuracy']
    S      = severity_score(psi, ksp, acc)
    label  = severity_label(S)
    action = label.split('(')[1].rstrip(')')
    print(f"  {yr}: PSI={psi:.4f}  KS_p={ksp:.4f}  Acc={acc:.3f}  "
          f"S={S:.4f}  →  {label}")
    severity_records.append({
        'year'          : yr,
        'psi'           : round(psi, 4),
        'ks_pvalue_mean': round(ksp, 4),
        'ks_drift_frac' : round(ks_drift_fraction[yr], 4),
        'accuracy'      : round(acc, 4),
        'adwin_changes' : adwin_results[yr]['n_changes'],
        'severity_score': round(S, 4),
        'severity_label': label,
        'recommended_action': action,
    })

In [ ]:
# ── Cell 9: Plot – Drift Severity Over Time ───────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Data Drift Analysis — Amazon Electronics Reviews (2019–2023)",
             fontsize=14, fontweight='bold', y=1.01)

In [ ]:
scores = [r['severity_score'] for r in severity_records]
psis   = [r['psi']             for r in severity_records]
accs   = [r['accuracy']        for r in severity_records]
ksf    = [r['ks_drift_frac']   for r in severity_records]

In [ ]:
# Color bars by severity tier
def bar_color(s):
    if s < 0.33: return '#27ae60'
    elif s < 0.66: return '#f39c12'
    else: return '#e74c3c'

In [ ]:
colors = [bar_color(s) for s in scores]

In [ ]:
# Panel 1: Severity Score
ax = axes[0, 0]
bars = ax.bar(YEARS, scores, color=colors, edgecolor='white', linewidth=0.8)
ax.axhline(0.33, color='#f39c12', linestyle='--', linewidth=1.2, label='Medium threshold (0.33)')
ax.axhline(0.66, color='#e74c3c', linestyle='--', linewidth=1.2, label='High threshold (0.66)')
for bar, s in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{s:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title("Composite Severity Score", fontweight='bold')
ax.set_ylabel("Severity Score (S)")
ax.set_ylim(0, 1.0)
ax.legend(fontsize=8)
ax.set_xticks(YEARS)

In [ ]:
# Panel 2: PSI Score
ax = axes[0, 1]
ax.plot(YEARS, psis, 'o-', color='#2980b9', linewidth=2.2, markersize=7)
ax.fill_between(YEARS, psis, alpha=0.15, color='#2980b9')
ax.axhline(0.10, color='#f39c12', linestyle=':', linewidth=1.2, label='Moderate (0.10)')
ax.axhline(0.20, color='#e74c3c', linestyle=':', linewidth=1.2, label='Significant (0.20)')
ax.set_title("PSI — Embedding Distribution Shift", fontweight='bold')
ax.set_ylabel("Population Stability Index")
ax.legend(fontsize=8)
ax.set_xticks(YEARS)

In [ ]:
# Panel 3: Accuracy over time
ax = axes[1, 0]
ax.plot(YEARS, accs, 's-', color='#8e44ad', linewidth=2.2, markersize=7, label='Window accuracy')
ax.axhline(BASELINE_ACC, color='#27ae60', linestyle='--', linewidth=1.5, label=f'Baseline ({BASELINE_ACC:.3f})')
ax.fill_between(YEARS,
                [BASELINE_ACC]*len(YEARS), accs,
                where=[a < BASELINE_ACC for a in accs],
                alpha=0.2, color='#e74c3c', label='Accuracy gap')
ax.set_title("Model Accuracy Degradation", fontweight='bold')
ax.set_ylabel("Accuracy")
ax.set_ylim(0.70, 0.92)
ax.legend(fontsize=8)
ax.set_xticks(YEARS)

In [ ]:
# Panel 4: KS drifted feature fraction
ax = axes[1, 1]
ax.bar(YEARS, [k*100 for k in ksf], color='#16a085', edgecolor='white', linewidth=0.8)
ax.set_title("KS Test — % Features with Drift (p<Bonferroni)", fontweight='bold')
ax.set_ylabel("% Features Drifted")
ax.set_ylim(0, 100)
ax.set_xticks(YEARS)
for i, (yr, k) in enumerate(zip(YEARS, ksf)):
    ax.text(yr, k*100 + 1, f'{k*100:.1f}%', ha='center', fontsize=9)

In [ ]:
# Legend patches for severity colour scheme
legend_patches = [
    mpatches.Patch(color='#27ae60', label='Low — Monitor'),
    mpatches.Patch(color='#f39c12', label='Medium — Partial Retrain'),
    mpatches.Patch(color='#e74c3c', label='High — Full Retrain'),
]
fig.legend(handles=legend_patches, loc='lower center',
           ncol=3, fontsize=9, frameon=True,
           bbox_to_anchor=(0.5, -0.04))

In [ ]:
plt.tight_layout()
plt.savefig('outputs/drift_severity_over_time.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n✓ Plot saved: outputs/drift_severity_over_time.png")

In [ ]:
# ── Cell 10: Save Results ─────────────────────────────────────────────────────
df_drift = pd.DataFrame(severity_records)
df_drift.to_csv('outputs/drift_results.csv', index=False)
print("✓ Saved: outputs/drift_results.csv")
print(df_drift.to_string(index=False))